In [ ]:

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    signup_date DATE,
    country TEXT
);

CREATE TABLE subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    plan TEXT,
    monthly_price REAL,
    start_date DATE,
    end_date DATE,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    subscription_id INTEGER,
    payment_date DATE,
    amount REAL,
    payment_status TEXT,
    FOREIGN KEY (subscription_id)
        REFERENCES subscriptions(subscription_id)
);
""")

print("Tables created successfully.")

Tables created successfully.


In [ ]:

conn.executescript("""
INSERT INTO customers VALUES
(1, '2026-01-05', 'USA'),
(2, '2026-01-12', 'Canada'),
(3, '2026-01-20', 'USA'),
(4, '2026-02-02', 'UK'),
(5, '2026-02-10', 'USA'),
(6, '2026-02-18', 'Canada'),
(7, '2026-03-01', 'UK'),
(8, '2026-03-08', 'USA');

INSERT INTO subscriptions VALUES
(101, 1, 'Basic',       10, '2026-01-05', NULL,         'active'),
(102, 2, 'Premium',    30, '2026-01-12', NULL,         'active'),
(103, 3, 'Basic',       10, '2026-01-20', '2026-02-20', 'cancelled'),
(104, 4, 'Standard',   20, '2026-02-02', NULL,         'active'),
(105, 5, 'Premium',    30, '2026-02-10', '2026-03-10', 'cancelled'),
(106, 6, 'Basic',       10, '2026-02-18', NULL,         'active'),
(107, 7, 'Standard',   20, '2026-03-01', NULL,         'active'),
(108, 8, 'Premium',    30, '2026-03-08', NULL,         'active');

INSERT INTO payments VALUES
(1001, 101, '2026-01-05', 10, 'paid'),
(1002, 102, '2026-01-12', 30, 'paid'),
(1003, 103, '2026-01-20', 10, 'paid'),
(1004, 101, '2026-02-05', 10, 'paid'),
(1005, 102, '2026-02-12', 30, 'paid'),
(1006, 103, '2026-02-20', 10, 'paid'),
(1007, 104, '2026-02-02', 20, 'paid'),
(1008, 105, '2026-02-10', 30, 'paid'),
(1009, 106, '2026-02-18', 10, 'paid'),
(1010, 101, '2026-03-05', 10, 'paid'),
(1011, 102, '2026-03-12', 30, 'paid'),
(1012, 104, '2026-03-02', 20, 'paid'),
(1013, 105, '2026-03-10', 30, 'paid'),
(1014, 106, '2026-03-18', 10, 'paid'),
(1015, 107, '2026-03-01', 20, 'paid'),
(1016, 108, '2026-03-08', 30, 'paid');
""")

print("Sample data inserted successfully.")

Sample data inserted successfully.


In [ ]:
def run_query(sql):
    return pd.read_sql_query(sql, conn)

In [ ]:
run_query("""
SELECT
    COUNT(*) AS payment_count,
    SUM(amount) AS total_revenue,
    AVG(amount) AS average_payment
FROM payments
WHERE payment_status = 'paid';
""")

,payment_count,total_revenue,average_payment
0,16,310.0,19.375


In [ ]:
run_query("""
SELECT
    strftime('%Y-%m', payment_date) AS payment_month,
    SUM(amount) AS monthly_revenue
FROM payments
WHERE payment_status = 'paid'
GROUP BY strftime('%Y-%m', payment_date)
ORDER BY payment_month ASC;
""")

,payment_month,monthly_revenue
0,2026-01,50.0
1,2026-02,110.0
2,2026-03,150.0


In [ ]:
run_query("""
SELECT
    s.plan,
    SUM(p.amount) AS total_revenue
FROM payments p
JOIN subscriptions s
    ON p.subscription_id = s.subscription_id
WHERE p.payment_status = 'paid'
GROUP BY s.plan
ORDER BY total_revenue DESC;
""")

,plan,total_revenue
0,Premium,180.0
1,Basic,70.0
2,Standard,60.0


In [ ]:
run_query("""
SELECT
    plan,
    COUNT(*) AS subscription_count
FROM subscriptions
WHERE status = 'active'
GROUP BY plan
ORDER BY subscription_count DESC;
""")

,plan,subscription_count
0,Standard,2
1,Premium,2
2,Basic,2


In [ ]:
run_query("""
SELECT
    COUNT(*) AS total_subscriptions,
    SUM(
        CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END
    ) AS cancelled_subscriptions,
    100.0 * SUM(
        CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END
    ) / COUNT(*) AS churn_rate_pct
FROM subscriptions;
""")

,total_subscriptions,cancelled_subscriptions,churn_rate_pct
0,8,2,25.0


In [ ]:
run_query("""
WITH month_ends AS (
    SELECT '2026-01-31' AS month_end
    UNION ALL
    SELECT '2026-02-28'
    UNION ALL
    SELECT '2026-03-31'
)
SELECT
    m.month_end,
    SUM(s.monthly_price) AS mrr
FROM month_ends m
JOIN subscriptions s
    ON s.start_date <= m.month_end
    AND (s.end_date IS NULL OR s.end_date > m.month_end)
GROUP BY m.month_end
ORDER BY m.month_end;
""")

,month_end,mrr
0,2026-01-31,50.0
1,2026-02-28,100.0
2,2026-03-31,120.0


In [ ]:
run_query("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', payment_date) AS payment_month,
        SUM(amount) AS revenue
    FROM payments
    WHERE payment_status = 'paid'
    GROUP BY strftime('%Y-%m', payment_date)
),

active_customers AS (
    SELECT
        '2026-01' AS payment_month,
        COUNT(DISTINCT c.customer_id) AS active_count
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-01-31'
      AND (s.end_date IS NULL OR s.end_date > '2026-01-31')

    UNION ALL

    SELECT
        '2026-02',
        COUNT(DISTINCT c.customer_id)
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-02-28'
      AND (s.end_date IS NULL OR s.end_date > '2026-02-28')

    UNION ALL

    SELECT
        '2026-03',
        COUNT(DISTINCT c.customer_id)
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-03-31'
      AND (s.end_date IS NULL OR s.end_date > '2026-03-31')
)

SELECT
    r.payment_month,
    r.revenue,
    a.active_count,
    ROUND(r.revenue * 1.0 / a.active_count, 2) AS revenue_per_active_customer
FROM monthly_revenue r
JOIN active_customers a
    ON r.payment_month = a.payment_month
ORDER BY r.payment_month;
""")

,payment_month,revenue,active_count,revenue_per_active_customer
0,2026-01,50.0,3,16.67
1,2026-02,110.0,5,22.00
2,2026-03,150.0,6,25.00


In [ ]:
run_query("""
SELECT
    strftime('%Y-%m', end_date) AS cancellation_month,
    COUNT(subscription_id) AS cancelled_subscriptions
FROM subscriptions
WHERE status = 'cancelled'
GROUP BY strftime('%Y-%m', end_date)
ORDER BY cancellation_month;
""")

,cancellation_month,cancelled_subscriptions
0,2026-02,1
1,2026-03,1


In [ ]:
run_query("""
WITH plan_churn AS (
    SELECT
        plan,
        COUNT(*) AS total_subscriptions,
        SUM(
            CASE
                WHEN status = 'cancelled' THEN 1
                ELSE 0
            END
        ) AS cancelled_subscriptions
    FROM subscriptions
    GROUP BY plan
)

SELECT
    plan,
    total_subscriptions,
    cancelled_subscriptions,
    ROUND(
        100.0 * cancelled_subscriptions / total_subscriptions,
        2
    ) AS churn_rate_pct
FROM plan_churn
ORDER BY churn_rate_pct DESC;
""")

,plan,total_subscriptions,cancelled_subscriptions,churn_rate_pct
0,Basic,3,1,33.33
1,Premium,3,1,33.33
2,Standard,2,0,0.00


In [ ]:
run_query("""
SELECT
    c.customer_id,
    SUM(p.amount) AS total_revenue
FROM customers c
JOIN subscriptions s
    ON c.customer_id = s.customer_id
JOIN payments p
    ON s.subscription_id = p.subscription_id
WHERE p.payment_status = 'paid'
GROUP BY c.customer_id
ORDER BY total_revenue DESC;
""")

,customer_id,total_revenue
0,2,90.0
1,5,60.0
2,4,40.0
3,8,30.0
4,1,30.0
5,7,20.0
6,6,20.0
7,3,20.0


In [ ]:
run_query("""
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        SUM(p.amount) AS total_revenue
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN payments p
        ON s.subscription_id = p.subscription_id
    WHERE p.payment_status = 'paid'
    GROUP BY c.customer_id
)

SELECT
    customer_id,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM customer_revenue
ORDER BY revenue_rank;
""")

,customer_id,total_revenue,revenue_rank
0,2,90.0,1
1,5,60.0,2
2,4,40.0,3
3,1,30.0,4
4,8,30.0,4
5,3,20.0,6
6,6,20.0,6
7,7,20.0,6


In [ ]:
run_query("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', payment_date) AS payment_month,
        SUM(amount) AS revenue
    FROM payments
    WHERE payment_status = 'paid'
    GROUP BY strftime('%Y-%m', payment_date)
),

revenue_with_previous AS (
    SELECT
        payment_month,
        revenue,
        LAG(revenue) OVER (ORDER BY payment_month) AS previous_revenue
    FROM monthly_revenue
)

SELECT
    payment_month,
    revenue,
    previous_revenue,
    ROUND(
        (revenue - previous_revenue) * 100.0 / previous_revenue,
        2
    ) AS pct_change
FROM revenue_with_previous
ORDER BY payment_month;
""")

,payment_month,revenue,previous_revenue,pct_change
0,2026-01,50.0,NaN,NaN
1,2026-02,110.0,50.0,120.00
2,2026-03,150.0,110.0,36.36


In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    COUNT(DISTINCT previous.customer_id) AS retained_customers,
    COUNT(DISTINCT previous.customer_id) * 100.0 / 3 AS retention_rate_pct
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND previous.month = '2026-01'
    AND current.month = '2026-02';
""")

,retained_customers,retention_rate_pct
0,3,100.0


In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    previous.month AS previous_month,
    current.month AS current_month,
    previous.customer_id
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND current.month = '2026-02'
    AND previous.month = '2026-01'
ORDER BY previous.customer_id;
""")

,previous_month,current_month,customer_id
0,2026-01,2026-02,1
1,2026-01,2026-02,2
2,2026-01,2026-02,3


In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    COUNT(DISTINCT previous.customer_id) AS retained_customers,
    COUNT(DISTINCT previous.customer_id) * 100.0 / 3 AS retention_rate_pct
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND previous.month = '2026-01'
    AND current.month = '2026-02';
""")

,retained_customers,retention_rate_pct
0,3,100.0


In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    previous.month AS previous_month,
    current.month AS current_month,
    COUNT(DISTINCT previous.customer_id) AS retained_customers
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND current.month = strftime(
        '%Y-%m',
        date(previous.month || '-01', '+1 month')
    )
GROUP BY previous.month, current.month
ORDER BY previous.month;
""")

In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
),

retention AS (
    SELECT
        previous.month AS previous_month,
        current.month AS current_month,
        COUNT(DISTINCT previous.customer_id) AS retained_customers
    FROM monthly_active previous
    JOIN monthly_active current
        ON previous.customer_id = current.customer_id
        AND current.month = strftime(
            '%Y-%m',
            date(previous.month || '-01', '+1 month')
        )
    GROUP BY previous.month, current.month
)

SELECT
    previous_month,
    current_month,
    (
        SELECT COUNT(DISTINCT customer_id)
        FROM monthly_active
        WHERE month = retention.previous_month
    ) AS previous_customers,
    retained_customers,
    ROUND(
        retained_customers * 100.0 /
        (
            SELECT COUNT(DISTINCT customer_id)
            FROM monthly_active
            WHERE month = retention.previous_month
        ),
        2
    ) AS retention_rate_pct
FROM retention
ORDER BY previous_month;
""")


,previous_month,current_month,previous_customers,retained_customers,retention_rate_pct
0,2026-01,2026-02,3,3,100.00
1,2026-02,2026-03,6,5,83.33


Business insight: Customer retention decreased from 100% in January → February to 66.67% in February → March. This means 3 of 3 January-active customers returned in February, while only 4 of 6 February-active customers returned in March. The business could investigate why 2 February customers did not remain active in March